In [150]:
import pandas as pd
import numpy as np
import os

In [151]:
weekly_selections = pd.read_csv('./FFS_manager_weekly_selections.csv')
team_weekly_data = pd.read_csv('./fpl_team_weekly_data.csv')
player_data = pd.read_csv('./data/merged_player_data.csv')

In [152]:
player_weekly_data = pd.read_csv('./fpl_player_weekly_data.csv')
managers = player_weekly_data[player_weekly_data['Position'] == 'Manager']
player_weekly_data = player_weekly_data[player_weekly_data['Position'] != 'Manager']
# managers#[['First Name', 'Last Name', 'Position', 'element']]

feats = [
    'element', 'fixture', 'opponent_team', 'total_points', 'was_home', 'team_h_score', 'team_a_score', 'round', 'minutes', 'goals_scored', 'assists',
    'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence',
    'creativity', 'threat', 'ict_index', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'value',
    'transfers_balance', 'selected', 'transfers_in', 'transfers_out', 'player_id', 'home_team_difficulty', 'away_team_difficulty', 'First Name',
    'Last Name', 'Team', 'Position',
]

player_weekly_data = player_weekly_data[feats]
player_weekly_data

,element,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,round,minutes,goals_scored,...,selected,transfers_in,transfers_out,player_id,home_team_difficulty,away_team_difficulty,First Name,Last Name,Team,Position
0,1,2,20,0,True,2,0,1,0,0,...,2923,0,0,1,3,5,Fábio,Ferreira Vieira,Arsenal,Midfielder
1,1,11,2,0,False,0,2,2,0,0,...,2321,84,874,1,5,4,Fábio,Ferreira Vieira,Arsenal,Midfielder
2,1,21,5,0,True,1,1,3,0,0,...,2397,355,634,1,3,5,Fábio,Ferreira Vieira,Arsenal,Midfielder
3,1,39,18,0,False,0,1,4,0,0,...,1650,0,747,1,5,3,Fábio,Ferreira Vieira,Arsenal,Midfielder
4,1,47,13,0,False,2,2,5,0,0,...,1494,0,174,1,5,4,Fábio,Ferreira Vieira,Arsenal,Midfielder
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27600,784,340,11,0,True,3,0,34,0,0,...,5498,509,523,784,1,3,Mateus,Mané,Wolves,Forward
27601,784,349,13,0,False,1,0,35,0,0,...,5327,323,346,784,3,4,Mateus,Mané,Wolves,Forward
27602,784,360,5,1,True,0,2,36,1,0,...,5431,361,371,784,3,3,Mateus,Mané,Wolves,Forward
27603,784,366,7,0,False,4,2,37,0,0,...,5639,350,237,784,3,3,Mateus,Mané,Wolves,Forward


## Add other data


In [153]:
teams = pd.read_csv('./2024-25/teams.csv')
fpl_id_to_team_lookup = dict(zip(teams['id'], teams['name']))

player_weekly_data = player_weekly_data.rename(columns={
    'First Name': 'first_name', 'Last Name': 'last_name', 'Team': 'team', 'Position': 'position'})



# copy of player_weekly_data
player_weekly_data_copy = player_weekly_data.copy()

# Map opponent_team, h_team, and a_team to team IDs using the lookup dictionary
player_weekly_data_copy['opponent_team'] = player_weekly_data_copy['opponent_team'].map(fpl_id_to_team_lookup)

# Cater for loaned players by mapping their team IDs to the correct team names
# Cast both columns to object/string type to avoid dtype incompatibility
player_weekly_data_copy[['team', 'opponent_team']] = player_weekly_data_copy[['team', 'opponent_team']].astype('object')
# Update both columns simultaneously in a single operation
player_weekly_data_copy.loc[2489, ['team', 'opponent_team']] = ['Aston Villa', 'Man Utd'] # Marcus Rashford
player_weekly_data_copy.loc[2419, ['team', 'opponent_team']] = ['Aston Villa', 'Chelsea'] # Axel Disasi
player_weekly_data_copy.loc[7471, ['team', 'opponent_team']] = ['Crystal Palace', 'Chelsea'] # Trevor Chalobah
player_weekly_data_copy.loc[[8841, 8858], ['team', 'opponent_team']] = [['Crystal Palace', 'Chelsea'],['Crystal Palace', 'Chelsea']] # Ben Chilwell
player_weekly_data_copy.loc[10927, ['team', 'opponent_team']] = ['Everton', 'Southampton'] # Carlos Acaraz  Durán
player_weekly_data_copy.loc[[12440, 12457], ['team', 'opponent_team']] = [['Ipswich', 'Brighton'],['Ipswich', 'Brighton']] # Julio Enciso
player_weekly_data_copy.loc[13506, ['team', 'opponent_team']] = ['Ipswich', 'Aston Villa'] # Jaden Philogene
player_weekly_data_copy.loc[14867, ['team', 'opponent_team']] = ['Leicester', 'Spurs'] # Oliver Skipp
player_weekly_data_copy.loc[18729, ['team', 'opponent_team']] = ['Arsenal', 'Man Utd'] # Ayden Heaven
player_weekly_data_copy.loc[21607, ['team', 'opponent_team']] = ['Southampton', 'Newcastle'] # Ryan Fraser
player_weekly_data_copy.loc[24551, ['team', 'opponent_team']] = ['West Ham', 'Brighton'] # Evan Ferguson
player_weekly_data_copy.loc[25304, ['team', 'opponent_team']] = ["Nott'm Forest", 'West Ham'] # James Ward-Prowse


player_weekly_data_copy[player_weekly_data_copy['opponent_team'] == player_weekly_data_copy['team']][['fixture', 'first_name', 'last_name', 'was_home', 'team', 'opponent_team', 'round']]

,fixture,first_name,last_name,was_home,team,opponent_team,round
2435,123,Axel,Disasi,True,Aston Villa,Aston Villa,13
2505,62,Marcus,Rashford,False,Aston Villa,Aston Villa,7
7551,194,Trevoh,Chalobah,True,Chelsea,Chelsea,20
8937,23,Ben,Chilwell,True,Crystal Palace,Crystal Palace,3
8954,194,Ben,Chilwell,False,Crystal Palace,Crystal Palace,20
11039,98,Carlos,Alcaraz Durán,True,Everton,Everton,10
12585,33,Julio,Enciso,True,Ipswich,Ipswich,4
12602,204,Julio,Enciso,False,Ipswich,Ipswich,21
13651,56,Jaden,Philogene,False,Ipswich,Ipswich,6
15028,10,Oliver,Skipp,False,Leicester,Leicester,1


In [154]:
# If was_home is True, set team to the home team; otherwise, set it to the away team
player_weekly_data_copy['h_team'] = np.where(player_weekly_data_copy['was_home'], player_weekly_data_copy['team'], player_weekly_data_copy['opponent_team'])
player_weekly_data_copy['a_team'] = np.where(player_weekly_data_copy['was_home'], player_weekly_data_copy['opponent_team'], player_weekly_data_copy['team'])

# Change home_team_difficulty and away_team_difficulty to team_difficult and opponent_team_difficulty respectively
player_weekly_data_copy['difficulty'] = np.where(player_weekly_data_copy['was_home'], player_weekly_data_copy['home_team_difficulty'], player_weekly_data_copy['away_team_difficulty'])
player_weekly_data_copy['opponent_difficulty'] = np.where(player_weekly_data_copy['was_home'], player_weekly_data_copy['away_team_difficulty'], player_weekly_data_copy['home_team_difficulty'])


player_weekly_data_copy[['was_home', 'team', 'opponent_team', 'h_team', 'a_team', 'round']]

,was_home,team,opponent_team,h_team,a_team,round
0,True,Arsenal,Wolves,Arsenal,Wolves,1
1,False,Arsenal,Aston Villa,Aston Villa,Arsenal,2
2,True,Arsenal,Brighton,Arsenal,Brighton,3
3,False,Arsenal,Spurs,Spurs,Arsenal,4
4,False,Arsenal,Man City,Man City,Arsenal,5
...,...,...,...,...,...,...
27600,True,Wolves,Leicester,Wolves,Leicester,34
27601,False,Wolves,Man City,Man City,Wolves,35
27602,True,Wolves,Brighton,Wolves,Brighton,36
27603,False,Wolves,Crystal Palace,Crystal Palace,Wolves,37


### Add Odds


In [155]:
odds = pd.read_csv('./E0 24-25.csv')
odds

,Div,Date,Time,h_team,a_team,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,16/08/2024,20:00,Manchester United,Fulham,1,0,H,0,0,...,1.86,2.07,1.83,2.11,1.88,2.11,1.82,2.05,1.90,2.08
1,E0,17/08/2024,12:30,Ipswich,Liverpool,0,2,A,0,0,...,2.05,1.88,2.04,1.90,2.20,2.00,1.99,1.88,2.04,1.93
2,E0,17/08/2024,15:00,Arsenal,Wolverhampton Wanderers,2,0,H,1,0,...,2.02,1.91,2.00,1.90,2.05,1.93,1.99,1.87,2.02,1.96
3,E0,17/08/2024,15:00,Everton,Brighton,0,3,A,0,1,...,1.87,2.06,1.86,2.07,1.92,2.10,1.83,2.04,1.88,2.11
4,E0,17/08/2024,15:00,Newcastle United,Southampton,1,0,H,1,0,...,1.87,2.06,1.88,2.06,1.89,2.10,1.82,2.05,1.89,2.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,E0,25/05/2025,16:00,Newcastle United,Everton,0,1,A,0,0,...,2.00,1.85,2.01,1.90,2.01,1.95,1.95,1.91,1.93,2.05
376,E0,25/05/2025,16:00,Nottingham Forest,Chelsea,0,1,A,0,0,...,1.80,2.05,1.86,2.08,1.86,2.08,1.81,2.05,1.86,2.14
377,E0,25/05/2025,16:00,Southampton,Arsenal,1,2,A,0,1,...,2.03,1.83,2.04,1.87,2.07,1.87,2.03,1.83,2.06,1.89
378,E0,25/05/2025,16:00,Tottenham,Brighton,1,4,A,1,0,...,1.95,1.90,2.00,1.93,2.01,1.93,1.95,1.89,2.06,1.93


In [156]:
odds_to_fpl_team_name_mapping = {
    'Arsenal': 'Arsenal',
    'Aston Villa': 'Aston Villa',
    'Bournemouth': 'Bournemouth',
    'Brentford': 'Brentford',
    'Brighton': 'Brighton',
    'Chelsea': 'Chelsea',
    'Crystal Palace': 'Crystal Palace',
    'Everton': 'Everton',
    'Fulham': 'Fulham',
    'Ipswich': 'Ipswich',
    'Leicester': 'Leicester',
    'Liverpool': 'Liverpool',
    'Manchester City': 'Man City',
    'Manchester United': 'Man Utd',
    'Newcastle United': 'Newcastle',
    "Nottingham Forest": "Nott'm Forest",
    'Southampton': 'Southampton',
    'Tottenham': 'Spurs',
    'West Ham': 'West Ham',
    'Wolverhampton Wanderers': 'Wolves'
}

# Copy of odds DataFrame to avoid modifying the original
odds_copy = odds.copy()

# Map the team names in the odds DataFrame to FPL team names using the mapping dictionary
odds_copy['h_team'] = odds_copy['h_team'].map(odds_to_fpl_team_name_mapping)
odds_copy['a_team'] = odds_copy['a_team'].map(odds_to_fpl_team_name_mapping)
odds_copy

,Div,Date,Time,h_team,a_team,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,16/08/2024,20:00,Man Utd,Fulham,1,0,H,0,0,...,1.86,2.07,1.83,2.11,1.88,2.11,1.82,2.05,1.90,2.08
1,E0,17/08/2024,12:30,Ipswich,Liverpool,0,2,A,0,0,...,2.05,1.88,2.04,1.90,2.20,2.00,1.99,1.88,2.04,1.93
2,E0,17/08/2024,15:00,Arsenal,Wolves,2,0,H,1,0,...,2.02,1.91,2.00,1.90,2.05,1.93,1.99,1.87,2.02,1.96
3,E0,17/08/2024,15:00,Everton,Brighton,0,3,A,0,1,...,1.87,2.06,1.86,2.07,1.92,2.10,1.83,2.04,1.88,2.11
4,E0,17/08/2024,15:00,Newcastle,Southampton,1,0,H,1,0,...,1.87,2.06,1.88,2.06,1.89,2.10,1.82,2.05,1.89,2.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,E0,25/05/2025,16:00,Newcastle,Everton,0,1,A,0,0,...,2.00,1.85,2.01,1.90,2.01,1.95,1.95,1.91,1.93,2.05
376,E0,25/05/2025,16:00,Nott'm Forest,Chelsea,0,1,A,0,0,...,1.80,2.05,1.86,2.08,1.86,2.08,1.81,2.05,1.86,2.14
377,E0,25/05/2025,16:00,Southampton,Arsenal,1,2,A,0,1,...,2.03,1.83,2.04,1.87,2.07,1.87,2.03,1.83,2.06,1.89
378,E0,25/05/2025,16:00,Spurs,Brighton,1,4,A,1,0,...,1.95,1.90,2.00,1.93,2.01,1.93,1.95,1.89,2.06,1.93


In [157]:
# Merge the odds DataFrame with player_weekly_data_copy on h_team, a_team
player_weekly_data_odds = pd.merge(
    player_weekly_data_copy,
    odds_copy[['h_team', 'a_team', 'B365H', 'B365D', 'B365A']],
    on=['h_team', 'a_team'],
    how='left'
)

player_weekly_data_odds[['h_team', 'a_team', 'B365H', 'B365D', 'B365A']]

,h_team,a_team,B365H,B365D,B365A
0,Arsenal,Wolves,1.18,7.50,13.00
1,Aston Villa,Arsenal,4.33,3.80,1.75
2,Arsenal,Brighton,1.33,5.50,8.50
3,Spurs,Arsenal,3.00,3.50,2.30
4,Man City,Arsenal,1.80,3.60,4.50
...,...,...,...,...,...
27278,Wolves,Leicester,1.50,4.20,6.25
27279,Man City,Wolves,1.40,4.75,7.50
27280,Wolves,Brighton,2.70,3.50,2.50
27281,Crystal Palace,Wolves,2.25,3.30,3.20


In [158]:
# Extract the relevant data values
# Convert the data to probabilities

def odds_to_probabilities(row):
    WHH = round(1/row['B365H'], 5)
    WHD = round(1/row['B365D'], 5)
    WHA = round(1/row['B365A'], 5)

    # Normalize the probabilities (to make the probabilities sum to 100%)
    WH_sum = WHH + WHD + WHA
    WHH_ = round(WHH/WH_sum, 3)
    WHD_ = round(WHD/WH_sum, 3)
    WHA_ = round(WHA/WH_sum, 3)

    return pd.Series([WHH_, WHD_, WHA_], index=['home_win_prob', 'draw_prob', 'away_win_prob'])

odds_prob_cols  = player_weekly_data_odds.apply(odds_to_probabilities, axis=1)
odds_prob_cols.columns = ['home_win_prob', 'draw_prob', 'away_win_prob']

# 2. Join the original DataFrame with the new columns all at once
player_weekly_data_probs = pd.concat([player_weekly_data_odds, odds_prob_cols], axis=1)

player_weekly_data_probs['win_prob'] = np.where(player_weekly_data_probs['was_home'], player_weekly_data_probs['home_win_prob'], player_weekly_data_probs['away_win_prob'])
player_weekly_data_probs['draw_prob'] = player_weekly_data_probs['draw_prob']
player_weekly_data_probs['lose_prob'] = np.where(player_weekly_data_probs['was_home'], player_weekly_data_probs['away_win_prob'], player_weekly_data_probs['home_win_prob'])

player_weekly_data_probs#[['was_home','home_win_prob', 'draw_prob', 'away_win_prob', 'win_prob', 'draw_prob', 'lose_prob']]

,element,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,round,minutes,goals_scored,...,difficulty,opponent_difficulty,B365H,B365D,B365A,home_win_prob,draw_prob,away_win_prob,win_prob,lose_prob
0,1,2,Wolves,0,True,2,0,1,0,0,...,3,5,1.18,7.50,13.00,0.801,0.126,0.073,0.801,0.073
1,1,11,Aston Villa,0,False,0,2,2,0,0,...,4,5,4.33,3.80,1.75,0.217,0.247,0.536,0.536,0.217
2,1,21,Brighton,0,True,1,1,3,0,0,...,3,5,1.33,5.50,8.50,0.715,0.173,0.112,0.715,0.112
3,1,39,Spurs,0,False,0,1,4,0,0,...,3,5,3.00,3.50,2.30,0.316,0.271,0.413,0.413,0.316
4,1,47,Man City,0,False,2,2,5,0,0,...,4,5,1.80,3.60,4.50,0.526,0.263,0.211,0.211,0.526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27278,784,340,Leicester,0,True,3,0,34,0,0,...,1,3,1.50,4.20,6.25,0.626,0.224,0.150,0.626,0.150
27279,784,349,Man City,0,False,1,0,35,0,0,...,4,3,1.40,4.75,7.50,0.675,0.199,0.126,0.126,0.675
27280,784,360,Brighton,1,True,0,2,36,1,0,...,3,3,2.70,3.50,2.50,0.351,0.271,0.379,0.351,0.379
27281,784,366,Crystal Palace,0,False,4,2,37,0,0,...,3,3,2.25,3.30,3.20,0.419,0.286,0.295,0.295,0.419


### Percentage Net Transfer


In [159]:
def ownership_change(row):
    net_transfers = row['transfers_in'] - row['transfers_out']
    total_transfers = row['transfers_in'] + row['transfers_out']
    net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

    return net_transfers_pct

player_weekly_data_probs['percentage_net_transfers'] = player_weekly_data_probs.apply(ownership_change, axis=1)
player_weekly_data_probs = player_weekly_data_probs.drop_duplicates(subset=['element', 'round'], keep='first')
player_weekly_data_probs

,element,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,round,minutes,goals_scored,...,opponent_difficulty,B365H,B365D,B365A,home_win_prob,draw_prob,away_win_prob,win_prob,lose_prob,percentage_net_transfers
0,1,2,Wolves,0,True,2,0,1,0,0,...,5,1.18,7.50,13.00,0.801,0.126,0.073,0.801,0.073,0.000000
1,1,11,Aston Villa,0,False,0,2,2,0,0,...,5,4.33,3.80,1.75,0.217,0.247,0.536,0.536,0.217,-0.824635
2,1,21,Brighton,0,True,1,1,3,0,0,...,5,1.33,5.50,8.50,0.715,0.173,0.112,0.715,0.112,-0.282103
3,1,39,Spurs,0,False,0,1,4,0,0,...,5,3.00,3.50,2.30,0.316,0.271,0.413,0.413,0.316,-1.000000
4,1,47,Man City,0,False,2,2,5,0,0,...,5,1.80,3.60,4.50,0.526,0.263,0.211,0.211,0.526,-1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27278,784,340,Leicester,0,True,3,0,34,0,0,...,3,1.50,4.20,6.25,0.626,0.224,0.150,0.626,0.150,-0.013566
27279,784,349,Man City,0,False,1,0,35,0,0,...,3,1.40,4.75,7.50,0.675,0.199,0.126,0.126,0.675,-0.034380
27280,784,360,Brighton,1,True,0,2,36,1,0,...,3,2.70,3.50,2.50,0.351,0.271,0.379,0.351,0.379,-0.013661
27281,784,366,Crystal Palace,0,False,4,2,37,0,0,...,3,2.25,3.30,3.20,0.419,0.286,0.295,0.295,0.419,0.192504


### Add xP


In [160]:
xP_data = pd.read_csv(f'./data/vaastav/data/2024-25/gws/gw1.csv')
for i in range(2,39):
    xP_data = pd.concat([xP_data, pd.read_csv(f'./data/vaastav/data/2024-25/gws/gw{i}.csv')], ignore_index=True)
xP_data = xP_data.drop_duplicates(subset=['element', 'round'], keep='first')
xP_data

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,value,was_home,yellow_cards,mng_clean_sheets,mng_draw,mng_goals_scored,mng_loss,mng_underdog_draw,mng_underdog_win,mng_win
0,Alex Scott,MID,Bournemouth,1.6,0,0,11,0,12.8,77,...,50,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Carlos Miguel dos Santos Pereira,GK,Nott'm Forest,2.2,0,0,0,0,0.0,427,...,45,True,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Tomiyasu Takehiro,DEF,Arsenal,0.0,0,0,0,0,0.0,22,...,50,True,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Malcolm Ebiowei,MID,Crystal Palace,0.0,0,0,0,0,0.0,197,...,45,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Ben Brereton Díaz,MID,Southampton,1.0,0,0,-2,0,14.0,584,...,55,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27600,Ashley Young,DEF,Everton,3.3,0,0,23,1,24.8,238,...,42,False,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27601,Somto Boniface,DEF,Ipswich,-0.5,0,0,0,0,0.0,799,...,40,True,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27602,Simon Rusk,AM,Southampton,1.3,0,0,0,0,0.0,748,...,5,True,0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
27603,Arne Slot,AM,Liverpool,4.0,0,0,0,0,0.0,744,...,15,True,0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [161]:
player_data_xP = player_weekly_data_probs.merge(xP_data[['name', 'xP', 'element', 'round' ]], on=['element', 'round'], how='left')
player_data_xP

,element,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,round,minutes,goals_scored,...,B365D,B365A,home_win_prob,draw_prob,away_win_prob,win_prob,lose_prob,percentage_net_transfers,name,xP
0,1,2,Wolves,0,True,2,0,1,0,0,...,7.50,13.00,0.801,0.126,0.073,0.801,0.073,0.000000,Fábio Ferreira Vieira,2.3
1,1,11,Aston Villa,0,False,0,2,2,0,0,...,3.80,1.75,0.217,0.247,0.536,0.536,0.217,-0.824635,Fábio Ferreira Vieira,0.8
2,1,21,Brighton,0,True,1,1,3,0,0,...,5.50,8.50,0.715,0.173,0.112,0.715,0.112,-0.282103,Fábio Ferreira Vieira,0.0
3,1,39,Spurs,0,False,0,1,4,0,0,...,3.50,2.30,0.316,0.271,0.413,0.413,0.316,-1.000000,Fábio Ferreira Vieira,0.0
4,1,47,Man City,0,False,2,2,5,0,0,...,3.60,4.50,0.526,0.263,0.211,0.211,0.526,-1.000000,Fábio Ferreira Vieira,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26914,784,340,Leicester,0,True,3,0,34,0,0,...,4.20,6.25,0.626,0.224,0.150,0.626,0.150,-0.013566,Mateus Mané,0.0
26915,784,349,Man City,0,False,1,0,35,0,0,...,4.75,7.50,0.675,0.199,0.126,0.126,0.675,-0.034380,Mateus Mané,-0.5
26916,784,360,Brighton,1,True,0,2,36,1,0,...,3.50,2.50,0.351,0.271,0.379,0.351,0.379,-0.013661,Mateus Mané,0.2
26917,784,366,Crystal Palace,0,False,4,2,37,0,0,...,3.30,3.20,0.419,0.286,0.295,0.295,0.419,0.192504,Mateus Mané,0.2


### Add CBITR


In [162]:
olbauday_url = "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2024-2025/playermatchstats"
olbauday_data = pd.read_csv(f'{olbauday_url}/GW1/playermatchstats.csv')
olbauday_data['round'] = 1

for gw in range(2,39):
    gw_data = pd.read_csv(f'{olbauday_url}/GW{gw}/playermatchstats.csv')
    gw_data['round'] = gw

    olbauday_data = pd.concat([olbauday_data, gw_data], ignore_index=True)

olbauday_data = olbauday_data.rename(columns={'player_id': 'element'}).drop_duplicates(subset=['element', 'round'], keep='first')
olbauday_data.to_csv('olbauday_data_playermatchstats.csv', index=False)
olbauday_data

,element,match_id,minutes_played,goals,assists,total_shots,xg,xa,xgot,shots_on_target,...,ground_duels_won_percent,aerial_duels_won_percent,successful_dribbles_percent,tackles_won_percent,start_min,finish_min,team_goals_conceded,penalties_scored,penalties_missed,round
0,17,24-25-prem-arsenal-vs-wolverhampton-wanderers,80,1,1,5,0.35,0.37,0.50,3,...,44,0,0,100,0,80,0,0,0,1
1,4,24-25-prem-arsenal-vs-wolverhampton-wanderers,90,1,1,5,0.45,0.04,0.73,1,...,45,38,33,100,0,90,0,0,0,1
2,15,24-25-prem-arsenal-vs-wolverhampton-wanderers,90,0,0,0,0.00,0.00,0.00,0,...,0,0,0,0,0,90,0,0,0,1
3,20,24-25-prem-arsenal-vs-wolverhampton-wanderers,90,0,0,1,0.06,0.06,0.00,0,...,60,0,0,67,0,90,0,0,0,1
4,9,24-25-prem-arsenal-vs-wolverhampton-wanderers,90,0,0,1,0.08,0.19,0.00,0,...,58,67,50,50,0,90,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11562,95,24-25-prem-wolverhampton-wanderers-vs-brentford,10,0,0,0,0.00,0.06,0.00,0,...,0,0,0,0,80,91,0,0,0,38
11563,86,24-25-prem-wolverhampton-wanderers-vs-brentford,4,0,0,1,0.14,0.00,0.00,0,...,0,0,0,0,86,91,0,0,0,38
11564,556,24-25-prem-wolverhampton-wanderers-vs-brentford,1,0,0,0,0.00,0.00,0.00,0,...,0,0,0,0,91,91,0,0,0,38
11565,535,24-25-prem-wolverhampton-wanderers-vs-brentford,3,0,0,0,0.00,0.00,0.00,0,...,0,0,0,0,87,91,0,0,0,38


In [163]:
cols_to_add = [
    'element', 'interceptions', 'recoveries', 'blocks', 'clearances', 'tackles', 'total_shots', 'shots_on_target', 'successful_dribbles',
    'touches_opposition_box', 'touches', 'chances_created', 'goals_prevented', 'sweeper_actions', 'accurate_passes_percent', 'successful_dribbles_percent',
    'tackles_won_percent', 'round'
]

player_data_cbitr = player_data_xP.merge(olbauday_data[cols_to_add], on=['element', 'round'], how='left')

player_data_cbitr[cols_to_add] = player_data_cbitr[cols_to_add].fillna(0)
player_data_cbitr

,element,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,round,minutes,goals_scored,...,shots_on_target,successful_dribbles,touches_opposition_box,touches,chances_created,goals_prevented,sweeper_actions,accurate_passes_percent,successful_dribbles_percent,tackles_won_percent
0,1,2,Wolves,0,True,2,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,11,Aston Villa,0,False,0,2,2,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,21,Brighton,0,True,1,1,3,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,39,Spurs,0,False,0,1,4,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,47,Man City,0,False,2,2,5,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26914,784,340,Leicester,0,True,3,0,34,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26915,784,349,Man City,0,False,1,0,35,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26916,784,360,Brighton,1,True,0,2,36,1,0,...,0.0,0.0,0.0,6.0,0.0,0.0,0.0,80.0,0.0,0.0
26917,784,366,Crystal Palace,0,False,4,2,37,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Roll Data


In [164]:
rolling_data = player_data_cbitr.copy()

## Cater for the missing rounds before rolling
# 1. Get all unique players and all unique rounds
all_players = rolling_data['element'].unique()
all_rounds = range(rolling_data['round'].min(), rolling_data['round'].max() + 1)

# 2. Create a MultiIndex of every player x every round
multi_idx = pd.MultiIndex.from_product([all_players, all_rounds], names=['element', 'round'])

# 3. Reindex the dataframe
# This inserts "empty" rows for missing player/round combinations
rolling_data = rolling_data.set_index(['element', 'round']).reindex(multi_idx).reset_index()

# Fill statistical columns with 0
rolling_data = rolling_data.fillna(0)

# Sort to ensure chronological order for the rolling window
rolling_data = rolling_data.sort_values(['element', 'round'])

rolling_data

,element,round,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,minutes,goals_scored,...,shots_on_target,successful_dribbles,touches_opposition_box,touches,chances_created,goals_prevented,sweeper_actions,accurate_passes_percent,successful_dribbles_percent,tackles_won_percent
0,1,1,2.0,Wolves,0.0,True,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,2,11.0,Aston Villa,0.0,False,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,3,21.0,Brighton,0.0,True,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,4,39.0,Spurs,0.0,False,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,5,47.0,Man City,0.0,False,2.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25075,804,34,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25076,804,35,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25077,804,36,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25078,804,37,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [165]:
# 1. Calculate the maximum 'selected' value across all players for each round
max_per_round = rolling_data.groupby('round')['selected'].max()

# 2. Shift the values down by 1 round (so Round 2 gets the max of Round 1, etc.)
prev_round_max = max_per_round.shift(1)
prev_round_total = np.floor(prev_round_max/0.80)
prev_round_total
# # 3. Map these 'previous round max' values back to the original DataFrame
rolling_data['total_managers'] = rolling_data['round'].map(prev_round_total)

rolling_data['selected_by_percent'] = (rolling_data['selected'] / rolling_data['total_managers'])
rolling_data['ownership_change'] = rolling_data.groupby('element')['selected'].diff()
rolling_data['percentage_net_transfers'] = rolling_data.groupby('element')['selected_by_percent'].diff()

In [166]:
rolling_data['points'] = rolling_data['total_points'] - rolling_data['bonus']
rolling_data

,element,round,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,minutes,goals_scored,...,chances_created,goals_prevented,sweeper_actions,accurate_passes_percent,successful_dribbles_percent,tackles_won_percent,total_managers,selected_by_percent,ownership_change,points
0,1,1,2.0,Wolves,0.0,True,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0
1,1,2,11.0,Aston Villa,0.0,False,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,6498771.0,0.000357,-602.0,0.0
2,1,3,21.0,Brighton,0.0,True,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,6907326.0,0.000347,76.0,0.0
3,1,4,39.0,Spurs,0.0,False,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,7568403.0,0.000218,-747.0,0.0
4,1,5,47.0,Man City,0.0,False,2.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,8660856.0,0.000173,-156.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25075,804,34,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,10069423.0,0.000000,0.0,0.0
25076,804,35,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,10278840.0,0.000000,0.0,0.0
25077,804,36,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,10169385.0,0.000000,0.0,0.0
25078,804,37,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,9806166.0,0.000000,0.0,0.0


In [167]:
rolling_features = [
    'points', 'xP', 'team_h_score', 'team_a_score', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals',
    'yellow_cards', 'red_cards', 'saves', 'influence', 'creativity', 'threat', 'ict_index', 'starts', 'expected_goals', 'expected_assists',
    'expected_goal_involvements', 'expected_goals_conceded', 'value',  'selected',

    'interceptions', 'recoveries', 'blocks', 'clearances', 'tackles', 'total_shots', 'chances_created', 'goals_prevented', 'sweeper_actions',
    'tackles_won_percent'
]

SPANS = [1,3,5]

# Sort to ensure chronological order for the rolling window
data_to_roll = rolling_data.sort_values(['element', 'round']).reset_index(drop=True)


#### Simple Averages


In [168]:
# data_to_roll = data_to_roll[data_to_roll['team'] != 'Mainz 05']
# Averagae values
new_features = {}
for col in rolling_features:
    # The rolling window now spans actual calendar rounds
    grp = data_to_roll.groupby('element')[col]

    for span in SPANS:
        new_features[f'{col}_rolling_{span}'] = grp.transform(
            lambda x: x.shift(1).rolling(span, min_periods=span).sum()/span
        )

# Join all new columns at once (no fragmentation)
data_to_roll = pd.concat(
    [data_to_roll, pd.DataFrame(new_features)],
    axis=1
)

data_to_roll

,element,round,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,minutes,goals_scored,...,chances_created_rolling_5,goals_prevented_rolling_1,goals_prevented_rolling_3,goals_prevented_rolling_5,sweeper_actions_rolling_1,sweeper_actions_rolling_3,sweeper_actions_rolling_5,tackles_won_percent_rolling_1,tackles_won_percent_rolling_3,tackles_won_percent_rolling_5
0,1,1,2.0,Wolves,0.0,True,2.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,11.0,Aston Villa,0.0,False,0.0,2.0,0.0,0.0,...,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN
2,1,3,21.0,Brighton,0.0,True,1.0,1.0,0.0,0.0,...,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN
3,1,4,39.0,Spurs,0.0,False,0.0,1.0,0.0,0.0,...,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN
4,1,5,47.0,Man City,0.0,False,2.0,2.0,0.0,0.0,...,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29787,804,34,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29788,804,35,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29789,804,36,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29790,804,37,0.0,0,0.0,0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Weighted Averages


In [169]:
ewm_features = {}
grouped = data_to_roll.groupby('element', sort=False)

# for col  in EWMA_SPANS:rollerolled_datad_datarolled_data
for col in rolling_features:
    shifted = grouped[col].shift(1)
    for span in SPANS:
        ewm_features[f'{col}_{span}_ewm'] = (
            shifted.groupby(data_to_roll['element'])
            .ewm(span=span, adjust=False)
            .mean()
            .reset_index(level=0, drop=True)
        )

data_to_roll = pd.concat([data_to_roll, pd.DataFrame(ewm_features)], axis=1)

# Remove fillna(0) values that were added
rolled_data = data_to_roll[data_to_roll['position'] != 0]
rolled_data.to_csv('./rolled_data_24_25.csv', index=False)
rolled_data

,element,round,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,minutes,goals_scored,...,chances_created_5_ewm,goals_prevented_1_ewm,goals_prevented_3_ewm,goals_prevented_5_ewm,sweeper_actions_1_ewm,sweeper_actions_3_ewm,sweeper_actions_5_ewm,tackles_won_percent_1_ewm,tackles_won_percent_3_ewm,tackles_won_percent_5_ewm
0,1,1,2.0,Wolves,0.0,True,2.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,11.0,Aston Villa,0.0,False,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,3,21.0,Brighton,0.0,True,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,4,39.0,Spurs,0.0,False,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,5,47.0,Man City,0.0,False,2.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29676,801,37,361.0,Newcastle,0.0,True,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29677,801,38,378.0,Southampton,0.0,False,1.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29715,802,38,371.0,Bournemouth,0.0,False,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29753,803,38,376.0,Newcastle,0.0,False,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Organize Manager Data


In [226]:
player_data = pd.read_csv('./rolled_data_24_25.csv')
player_data[player_data['round'] == 38]

,element,round,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,minutes,goals_scored,...,chances_created_5_ewm,goals_prevented_1_ewm,goals_prevented_3_ewm,goals_prevented_5_ewm,sweeper_actions_1_ewm,sweeper_actions_3_ewm,sweeper_actions_5_ewm,tackles_won_percent_1_ewm,tackles_won_percent_3_ewm,tackles_won_percent_5_ewm
36,1,38,378.0,Southampton,0.0,False,1.0,2.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
73,2,38,378.0,Southampton,0.0,False,1.0,2.0,0.0,0.0,...,0.000859,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000443,0.049716
110,3,38,378.0,Southampton,0.0,False,1.0,2.0,0.0,0.0,...,0.014462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.210903,1.731267
147,4,38,378.0,Southampton,1.0,False,1.0,2.0,19.0,0.0,...,0.002929,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001058,0.050103
184,5,38,378.0,Southampton,0.0,False,1.0,2.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26911,800,38,373.0,West Ham,0.0,True,1.0,3.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
26915,801,38,378.0,Southampton,0.0,False,1.0,2.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
26916,802,38,371.0,Bournemouth,0.0,False,2.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
26917,803,38,376.0,Newcastle,0.0,False,0.0,1.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000


In [ ]:
rolled_data

,element,round,fixture,opponent_team,total_points,was_home,team_h_score,team_a_score,minutes,goals_scored,...,chances_created_5_ewm,goals_prevented_1_ewm,goals_prevented_3_ewm,goals_prevented_5_ewm,sweeper_actions_1_ewm,sweeper_actions_3_ewm,sweeper_actions_5_ewm,tackles_won_percent_1_ewm,tackles_won_percent_3_ewm,tackles_won_percent_5_ewm
0,1,1,2.0,Wolves,0.0,True,2.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,11.0,Aston Villa,0.0,False,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,3,21.0,Brighton,0.0,True,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,4,39.0,Spurs,0.0,False,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,5,47.0,Man City,0.0,False,2.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29676,801,37,361.0,Newcastle,0.0,True,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29677,801,38,378.0,Southampton,0.0,False,1.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29715,802,38,371.0,Bournemouth,0.0,False,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29753,803,38,376.0,Newcastle,0.0,False,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [206]:
selections_df = (
    weekly_selections.groupby(['team_id', 'game_week'])['element']
    .apply(list)
    .reset_index(name='squad')
    .rename(columns={'game_week': 'round'})
)

selections_df

,team_id,round,squad
0,5,1,"[201, 44, 422, 18, 54, 181, 328, 13, 398, 351,..."
1,5,2,"[201, 162, 18, 461, 255, 181, 13, 328, 398, 35..."
2,5,3,"[201, 44, 18, 255, 54, 13, 19, 398, 328, 351, ..."
3,5,4,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,..."
4,5,5,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,..."
...,...,...,...
193776,9919919,34,"[413, 533, 579, 163, 71, 402, 514, 328, 566, 2..."
193777,9919919,35,"[15, 211, 579, 350, 328, 514, 99, 402, 207, 40..."
193778,9919919,36,"[15, 211, 579, 350, 345, 514, 99, 402, 110, 40..."
193779,9919919,37,"[15, 350, 211, 579, 514, 54, 99, 402, 345, 755..."


In [ ]:


# 1. Create a fast lookup dictionary (element ID -> Position)
element_pos_map = (
    player_data[['element', 'position']]
    .drop_duplicates(subset=['element'])
    .set_index('element')['position']
    .to_dict()
)

# 2. Extract position lists cleanly
positions = ['Midfielder', 'Forward', 'Defender', 'Goalkeeper']

for pos in positions:
    col_name = pos.lower() + 's'
    selections_df[col_name] = selections_df['squad'].apply(
        lambda squad: [player for player in squad if element_pos_map.get(player) == pos]
    )

selections_df

,team_id,round,squad,midfielders,forwards,defenders,goalkeepers
0,5,1,"[201, 44, 422, 18, 54, 181, 328, 13, 398, 351,...","[54, 181, 328, 13, 398]","[351, 401, 69]","[44, 422, 18, 255, 461]","[201, 209]"
1,5,2,"[201, 162, 18, 461, 255, 181, 13, 328, 398, 35...","[181, 13, 328, 398, 54]","[351, 401, 69]","[162, 18, 461, 255, 44]","[201, 209]"
2,5,3,"[201, 44, 18, 255, 54, 13, 19, 398, 328, 351, ...","[54, 13, 19, 398, 328]","[351, 401, 69]","[44, 18, 255, 162, 461]","[201, 209]"
3,5,4,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,...","[54, 136, 19, 398, 328]","[351, 401, 69]","[44, 18, 255, 162, 461]","[201, 209]"
4,5,5,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,...","[54, 136, 19, 398, 328]","[351, 401, 69]","[44, 18, 255, 461, 162]","[201, 209]"
...,...,...,...,...,...,...,...
193776,9919919,34,"[413, 533, 579, 163, 71, 402, 514, 328, 566, 2...","[71, 402, 514, 328, 168]","[566, 252, 401]","[533, 579, 163, 255, 70]","[413, 554]"
193777,9919919,35,"[15, 211, 579, 350, 328, 514, 99, 402, 207, 40...","[328, 514, 99, 402, 585]","[207, 401, 755]","[211, 579, 350, 18, 409]","[15, 513]"
193778,9919919,36,"[15, 211, 579, 350, 345, 514, 99, 402, 110, 40...","[345, 514, 99, 402, 585]","[110, 401, 755]","[211, 579, 350, 18, 409]","[15, 513]"
193779,9919919,37,"[15, 350, 211, 579, 514, 54, 99, 402, 345, 755...","[514, 54, 99, 402, 345]","[755, 110, 401]","[350, 211, 579, 18, 409]","[15, 513]"


In [246]:
# positions_players

In [250]:
# Form a binary representation of selections for each position
positions = ['goalkeeper', 'defender', 'midfielder', 'forward']

gw_position_players = {}
for round in range(1, 39):
    if round == 1:
        curr_player_data = player_data[player_data['round'] ==  round]
    else:
        curr_player_data = player_data[player_data['round'] < round+1]

    positions_players = {}
    for p in positions:
        positions_players[p] = curr_player_data[curr_player_data['position'].str.lower() == p][['element']].drop_duplicates(subset=['element'])['element'].tolist()

    gw_position_players[round] = positions_players

# gw_position_players

In [253]:
players_by_round = player_weekly_data.groupby('round')['element'].apply(list).to_dict()

def weekly_selections_binary(row, pos):
    # print(row[pos.lower() + 's'])
    gw = row['round']

    positions_players = gw_position_players[gw]
    squad_set = set(row[pos.lower() + 's'])
    round_players = positions_players[pos]

    return [1 if player in squad_set else 0 for player in round_players]

for pos in positions_players.keys():
    selections_df[pos + '_binary'] = selections_df.apply(lambda row: weekly_selections_binary(row, pos), axis=1)

selections_df

,team_id,round,squad,midfielders,forwards,defenders,goalkeepers,goalkeeper_binary,defender_binary,midfielder_binary,forward_binary
0,5,1,"[201, 44, 422, 18, 54, 181, 328, 13, 398, 351,...","[54, 181, 328, 13, 398]","[351, 401, 69]","[44, 422, 18, 255, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
1,5,2,"[201, 162, 18, 461, 255, 181, 13, 328, 398, 35...","[181, 13, 328, 398, 54]","[351, 401, 69]","[162, 18, 461, 255, 44]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
2,5,3,"[201, 44, 18, 255, 54, 13, 19, 398, 328, 351, ...","[54, 13, 19, 398, 328]","[351, 401, 69]","[44, 18, 255, 162, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
3,5,4,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,...","[54, 136, 19, 398, 328]","[351, 401, 69]","[44, 18, 255, 162, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
4,5,5,"[201, 44, 18, 255, 54, 136, 19, 398, 328, 351,...","[54, 136, 19, 398, 328]","[351, 401, 69]","[44, 18, 255, 461, 162]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...
193776,9919919,34,"[413, 533, 579, 163, 71, 402, 514, 328, 566, 2...","[71, 402, 514, 328, 168]","[566, 252, 401]","[533, 579, 163, 255, 70]","[413, 554]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
193777,9919919,35,"[15, 211, 579, 350, 328, 514, 99, 402, 207, 40...","[328, 514, 99, 402, 585]","[207, 401, 755]","[211, 579, 350, 18, 409]","[15, 513]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
193778,9919919,36,"[15, 211, 579, 350, 345, 514, 99, 402, 110, 40...","[345, 514, 99, 402, 585]","[110, 401, 755]","[211, 579, 350, 18, 409]","[15, 513]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
193779,9919919,37,"[15, 350, 211, 579, 514, 54, 99, 402, 345, 755...","[514, 54, 99, 402, 345]","[755, 110, 401]","[350, 211, 579, 18, 409]","[15, 513]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."


In [254]:
league_1 = pd.read_csv('./FFS_League_1.csv')
league_teams = league_1.loc[0:19,].copy()
league_team_ids = league_teams['Team id'].tolist()
league_selections_df = selections_df[selections_df['team_id'].isin(league_team_ids)].copy()
league_selections_df

,team_id,round,squad,midfielders,forwards,defenders,goalkeepers,goalkeeper_binary,defender_binary,midfielder_binary,forward_binary
304,205,1,"[201, 350, 231, 333, 328, 317, 181, 19, 351, 4...","[328, 317, 181, 19, 481]","[351, 401, 82]","[350, 231, 333, 255, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
305,205,2,"[201, 350, 255, 461, 328, 317, 181, 19, 351, 4...","[328, 317, 181, 19, 481]","[351, 401, 251]","[350, 255, 461, 333, 231]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
306,205,3,"[201, 350, 255, 231, 328, 317, 19, 54, 351, 40...","[328, 317, 19, 54, 481]","[351, 401, 251]","[350, 255, 231, 333, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
307,205,4,"[201, 311, 350, 255, 328, 317, 19, 54, 351, 40...","[328, 317, 19, 54, 481]","[351, 401, 251]","[311, 350, 255, 231, 461]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
308,205,5,"[201, 461, 350, 255, 311, 19, 328, 54, 251, 35...","[19, 328, 54, 317, 481]","[251, 351, 58]","[461, 350, 255, 311, 231]","[201, 209]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...,...
166815,4131447,34,"[185, 533, 418, 579, 163, 327, 182, 328, 252, ...","[327, 182, 328, 392, 247]","[252, 401, 541]","[533, 418, 579, 163, 70]","[185, 152]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
166816,4131447,35,"[15, 44, 350, 211, 99, 328, 199, 54, 447, 755,...","[99, 328, 199, 54, 762]","[447, 755, 207]","[44, 350, 211, 18, 361]","[15, 513]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
166817,4131447,36,"[513, 361, 211, 350, 99, 328, 199, 106, 447, 1...","[99, 328, 199, 106, 54]","[447, 110, 207]","[361, 211, 350, 44, 18]","[513, 15]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
166818,4131447,37,"[513, 361, 211, 350, 99, 106, 199, 328, 54, 11...","[99, 106, 199, 328, 54]","[110, 58, 447]","[361, 211, 350, 44, 18]","[513, 15]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."


In [255]:
league_selections_df.to_csv('./league_selections_df.csv', index=False)
